# Sprint 4 Runner (Colab)

Notebook khusus untuk menjalankan `Sprint 4 Plan v3` end-to-end dengan output log tampil penuh di setiap sel.

## 0) Settings

Isi `REPO_URL` dengan repository kamu. Kalau repo private, pastikan token/Git auth sudah siap di Colab. I

In [1]:
# Core settings
REPO_URL = 'https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git'  # contoh: https://github.com/<user>/<repo>.git
REPO_BRANCH = 'feat/sprint4-anomaly-first-low-fpr'  # branch yang mau dipakai di Colab
PROJECT_NAME = 'nids-cnn-lstm-autoencoder'
DRIVE_ROOT = '/content/drive/MyDrive/nids-cnn-lstm-autoencoder'

# Raw dataset source (default mengikuti Sprint 3)
# Jika folder ini tidak ada, notebook fallback ke {DRIVE_ROOT}/data/raw
RAW_DRIVE_SOURCE = '/content/drive/MyDrive/nids-data/raw'

FORCE_RECLONE = False

# Stage toggles (set True/False sesuai kebutuhan)
RUN_STAGE0 = True
RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_STAGE3_4 = True
RUN_SUMMARIZE = True


In [2]:
# Colab bootstrap + mount drive
import os
import sys
import json
import time
import shlex
import shutil
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


IN_COLAB = True
Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/nids-cnn-lstm-autoencoder


In [3]:
# Clone or update repo in /content + checkout branch
PROJECT_ROOT = Path('/content') / PROJECT_NAME

if FORCE_RECLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    if '<REPO_URL_HERE>' in REPO_URL:
        raise ValueError('Set REPO_URL dulu di cell Settings')
    cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

# Always sync and checkout selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--prune'], check=False)

# Try checkout branch; if branch only exists on origin, create tracking local branch.
ret = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_BRANCH], check=False)
if ret.returncode != 0:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '-b', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)

# Pull latest on selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', REPO_BRANCH], check=False)

active_branch = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], text=True).strip()
print('[GIT] active branch =', active_branch)


[CMD] git clone https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git /content/nids-cnn-lstm-autoencoder
PROJECT_ROOT = /content/nids-cnn-lstm-autoencoder
cwd = /content/nids-cnn-lstm-autoencoder
[GIT] active branch = feat/sprint4-anomaly-first-low-fpr


In [4]:
# Install dependencies (Colab-safe: keep built-in scientific stack)
from pathlib import Path
import importlib.util

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')


def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if IN_COLAB:
    # IMPORTANT:
    # Colab sudah punya numpy/pandas/scikit/tensorflow yang saling kompatibel.
    # Reinstall paket-paket ini sering memicu ABI mismatch (numpy.dtype size changed).
    print('[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).')

    # Install only lightweight utilities if missing.
    lightweight = [
        'pyyaml',
        'joblib',
        'seaborn',
    ]

    for pkg in lightweight:
        mod = 'yaml' if pkg == 'pyyaml' else pkg
        if importlib.util.find_spec(mod) is None:
            _run([sys.executable, '-m', 'pip', 'install', pkg])
        else:
            print(f'[INFO] {pkg} already available')

    print('\n[OK] Dependency step finished (Colab-safe mode).')
    print('[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.')
else:
    # Local/non-Colab: follow project requirements as usual.
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req)])


[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).
[INFO] pyyaml already available
[INFO] joblib already available
[INFO] seaborn already available

[OK] Dependency step finished (Colab-safe mode).
[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.


In [5]:
# Link data/model/results sprint4 to Drive (persist across disconnects)
os.chdir(PROJECT_ROOT)

raw_source = RAW_DRIVE_SOURCE if Path(RAW_DRIVE_SOURCE).exists() else f'{DRIVE_ROOT}/data/raw'
print(f'[RAW] using source: {raw_source}')

paths = [
    ('data/raw', raw_source),
    ('data/sprint4', f'{DRIVE_ROOT}/data/sprint4'),
    ('models/sprint4', f'{DRIVE_ROOT}/models/sprint4'),
    ('results/sprint4', f'{DRIVE_ROOT}/results/sprint4'),
]

for _, dst in paths:
    Path(dst).mkdir(parents=True, exist_ok=True)

for src, dst in paths:
    src_path = Path(src)
    if src_path.is_symlink() or src_path.exists():
        if src_path.is_symlink() or src_path.is_file():
            src_path.unlink()
        else:
            shutil.rmtree(src_path)
    src_path.parent.mkdir(parents=True, exist_ok=True)
    src_path.symlink_to(Path(dst), target_is_directory=True)
    print(f'[LINK] {src} -> {dst}')

print('[OK] Symlink setup complete')

# Fast preflight check
required_dirs = [
    Path('data/raw/CIC-IDS2017'),
    Path('data/raw/CSE-CIC-IDS2018'),
]
for d in required_dirs:
    print(f'[CHECK] {d}:', 'OK' if d.exists() else 'MISSING')


[LINK] data/raw -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/data/raw
[LINK] data/sprint4 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/data/sprint4
[LINK] models/sprint4 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/models/sprint4
[LINK] results/sprint4 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/results/sprint4
[OK] Symlink setup complete


In [6]:
# GPU check + stream helpers (all command output visible)
try:
    import tensorflow as tf
except Exception as e:
    print('[ERROR] TensorFlow import gagal:', repr(e))
    print('Kemungkinan besar environment ABI belum sinkron setelah pip install.')
    print('Solusi: Runtime > Restart runtime, lalu jalankan lagi dari cell GPU check ini.')
    raise

print('Python executable :', sys.executable)
print('TensorFlow        :', tf.__version__)
print('Built with CUDA   :', tf.test.is_built_with_cuda())
print('Built with GPU sup:', tf.test.is_built_with_gpu_support())
print('Physical GPU list :', tf.config.list_physical_devices('GPU'))
print('Logical GPU list  :', tf.config.list_logical_devices('GPU'))


def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'


def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts


def run_cmd_stream(title: str, cmd: str):
    print(f'[RUN] {title}')
    args = _normalize_cmd(cmd)
    print('[CMD]', ' '.join(args))
    t0 = time.time()

    proc = subprocess.Popen(
        args,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    last_line = ''
    for line in proc.stdout:
        print(line, end='')
        last_line = line.strip()

    ret = proc.wait()
    print(f'\n[EXIT CODE] {ret}')
    if ret != 0:
        raise RuntimeError(f"{title} failed (exit={ret}). Last line: {last_line}")

    print(f'[DONE] {title} in {_fmt_duration(time.time() - t0)}')


def run_stage(stage_name: str):
    run_cmd_stream(
        title=f'Sprint4 {stage_name}',
        cmd=f'python scripts/sprint4/research_runner.py --stage-names {stage_name} --skip-existing',
    )


Python executable : /usr/bin/python3
TensorFlow        : 2.19.0
Built with CUDA   : True
Built with GPU sup: True
Physical GPU list : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPU list  : [LogicalDevice(name='/device:GPU:0', device_type='GPU')]


## 1) Optional Dry-Run

Validasi command tanpa eksekusi real training/eval.

In [7]:
run_cmd_stream(
    title='Sprint4 Dry-Run Stage0',
    cmd='python scripts/sprint4/research_runner.py --dry-run --stage-names stage0 --no-summarize',
)


[RUN] Sprint4 Dry-Run Stage0
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --dry-run --stage-names stage0 --no-summarize
[RUN] s4_00_repro_v4_full | stage=stage0 | stages=['preprocess', 'train', 'eval'] | tag=s4_00_repro_v4_full
[CMD] /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_00_repro_v4_full.yaml
[CMD] /usr/bin/python3 scripts/sprint4/train.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_00_repro_v4_full.yaml --variant hybrid
[CMD] /usr/bin/python3 scripts/sprint4/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_00_repro_v4_full.yaml --model models/sprint4/s4_00_repro_v4_full/cnn_lstm_ae/best_model.keras --tag s4_00_repro_v4_full
[RUN] s4_01_strict_contract_full | stage=stage0 | stages=['eval'] | tag=s4_01_strict_contract_full
[CMD] /usr/bin/python3 scripts/sprint4/eval.py --config /content/nids-cnn-lstm-autoen

## 2) Execute Sprint 4 Stages

In [8]:
if RUN_STAGE0:
    run_stage('stage0')
else:
    print('[SKIP] stage0')


[RUN] Sprint4 stage0
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --stage-names stage0 --skip-existing
Traceback (most recent call last):
  File "/content/nids-cnn-lstm-autoencoder/scripts/preprocess.py", line 943, in <module>
    main()
  File "/content/nids-cnn-lstm-autoencoder/scripts/preprocess.py", line 666, in main
    raise FileNotFoundError(f"No CSV files found under {data_raw_cic}")
FileNotFoundError: No CSV files found under data/raw/CIC-IDS2017
Traceback (most recent call last):
  File "/content/nids-cnn-lstm-autoencoder/scripts/sprint4/preprocess.py", line 24, in <module>
    main()
  File "/content/nids-cnn-lstm-autoencoder/scripts/sprint4/preprocess.py", line 20, in main
    subprocess.run(cmd, cwd=str(ROOT), check=True)
  File "/usr/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['/usr/bin/python3', 'scripts/preprocess.py', '--config', '/content/nids-cnn-lstm-autoenc

In [9]:
if RUN_STAGE1:
    run_stage('stage1')
else:
    print('[SKIP] stage1')


[RUN] Sprint4 stage1
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --stage-names stage1 --skip-existing
Traceback (most recent call last):
  File "/content/nids-cnn-lstm-autoencoder/scripts/preprocess.py", line 943, in <module>
    main()
  File "/content/nids-cnn-lstm-autoencoder/scripts/preprocess.py", line 666, in main
    raise FileNotFoundError(f"No CSV files found under {data_raw_cic}")
FileNotFoundError: No CSV files found under data/raw/CIC-IDS2017
Traceback (most recent call last):
  File "/content/nids-cnn-lstm-autoencoder/scripts/sprint4/preprocess.py", line 24, in <module>
    main()
  File "/content/nids-cnn-lstm-autoencoder/scripts/sprint4/preprocess.py", line 20, in main
    subprocess.run(cmd, cwd=str(ROOT), check=True)
  File "/usr/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['/usr/bin/python3', 'scripts/preprocess.py', '--config', '/content/nids-cnn-lstm-autoenc

In [10]:
if RUN_STAGE2:
    run_stage('stage2')
else:
    print('[SKIP] stage2')


[RUN] Sprint4 stage2
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --stage-names stage2 --skip-existing
[SKIP] s4_m01_baseline_arch | Run s4_m01_baseline_arch requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m02_dropout_030 | Run s4_m02_dropout_030 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m03_latent_64 | Run s4_m03_latent_64 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m04_cnn_64_128_128 | Run s4_m04_cnn_64_128_128 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m05_lstm_192_96 | Run s4_m05_lstm_192_96 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m06_loss_huber | Run s4_m06_loss_huber requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s4_m07_loss_mse_mae_mix_a07 | Run s4_m07_loss_mse_mae_mix_a07 requires template_from_best_stage=stage1, but sou

In [11]:
if RUN_STAGE3_4:
    run_cmd_stream(
        title='Sprint4 stage3+stage4',
        cmd='python scripts/sprint4/research_runner.py --stage-names stage3,stage4 --skip-existing',
    )
else:
    print('[SKIP] stage3,stage4')


[RUN] Sprint4 stage3+stage4
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --stage-names stage3,stage4 --skip-existing
[SKIP] s4_t01_source_percentile_p93 | Run s4_t01_source_percentile_p93 requires template_from_best_stage=stage2, but source stage is inconclusive.
[SKIP] s4_t02_source_percentile_p95 | Run s4_t02_source_percentile_p95 requires template_from_best_stage=stage2, but source stage is inconclusive.
[SKIP] s4_t03_source_percentile_p97 | Run s4_t03_source_percentile_p97 requires template_from_best_stage=stage2, but source stage is inconclusive.
[SKIP] s4_t04_source_gaussian_k18 | Run s4_t04_source_gaussian_k18 requires template_from_best_stage=stage2, but source stage is inconclusive.
[SKIP] s4_t05_source_calib_f1 | Run s4_t05_source_calib_f1 requires template_from_best_stage=stage2, but source stage is inconclusive.
[SKIP] s4_t06_source_calib_guardrail_fpr010 | Run s4_t06_source_calib_guardrail_fpr010 requires template_from_best_stage=stage2, but source stage is in

In [12]:
if RUN_SUMMARIZE:
    run_cmd_stream(
        title='Sprint4 summarize-only',
        cmd='python scripts/sprint4/research_runner.py --summarize-only',
    )
else:
    print('[SKIP] summarize-only')


[RUN] Sprint4 summarize-only
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --summarize-only
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json

[EXIT CODE] 0
[DONE] Sprint4 summarize-only in 0s


## 3) Show Outputs (Summary, Gate, Report)

In [13]:
import pandas as pd
from IPython.display import display, Markdown

summary_path = PROJECT_ROOT / 'results/sprint4/summary.csv'
gate_path = PROJECT_ROOT / 'results/sprint4/gate_decision.json'
report_path = PROJECT_ROOT / 'docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md'
status_path = PROJECT_ROOT / 'results/sprint4/runtime/run_status.json'

print('summary_path =', summary_path)
print('gate_path    =', gate_path)
print('report_path  =', report_path)
print('status_path  =', status_path)

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print('\n[SUMMARY HEAD]')
    display(df.head(30))
    print('\n[STATUS COUNTS]')
    if 'terminal_status' in df.columns:
        display(df['terminal_status'].value_counts(dropna=False))
else:
    print('[WARN] summary.csv not found')

if gate_path.exists():
    gate = json.loads(gate_path.read_text(encoding='utf-8'))
    print('\n[GATE DECISION]')
    print(json.dumps(gate, indent=2))
else:
    print('[WARN] gate_decision.json not found')

if report_path.exists():
    print('\n[REPORT PREVIEW]')
    txt = report_path.read_text(encoding='utf-8')
    display(Markdown(txt[:8000]))
else:
    print('[WARN] report markdown not found')

if status_path.exists():
    print('\n[RUNTIME STATUS JSON]')
    print(status_path.read_text(encoding='utf-8')[:8000])


summary_path = /content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv
gate_path    = /content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json
report_path  = /content/nids-cnn-lstm-autoencoder/docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md
status_path  = /content/nids-cnn-lstm-autoencoder/results/sprint4/runtime/run_status.json

[SUMMARY HEAD]


,run_id,stage_name,profile,model_variant,active,condition,stages,tag,terminal_status,retry_count,...,cse_prec,cse_rec,cse_f1,cse_fpr,cse_auc,f1_gap,auc_gap,accuracy_gap,wall_time_sec,last_error
0,s4_00_repro_v4_full,stage0,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_00_repro_v4_full,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.57s: /usr/bin/pyth...
1,s4_01_strict_contract_full,stage0,s4_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s4_01_strict_contract_full,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 10.64s: /usr/bin/pyt...
2,s4_p01_robust_q995_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p01_robust_q995_clip20_corr090,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.52s: /usr/bin/pyth...
3,s4_p02_robust_q997_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p02_robust_q997_clip20_corr090,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.53s: /usr/bin/pyth...
4,s4_p03_quantile_q995_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p03_quantile_q995_clip20_corr090,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.53s: /usr/bin/pyth...
5,s4_p04_robust_q995_clip15_corr085,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p04_robust_q995_clip15_corr085,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.52s: /usr/bin/pyth...
6,s4_p05_robust_q995_clip20_corr095,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p05_robust_q995_clip20_corr095,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.51s: /usr/bin/pyth...
7,s4_p06_robust_q995_clip20_win20_stride2,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p06_robust_q995_clip20_win20_stride2,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Command failed rc=1 after 1.51s: /usr/bin/pyth...
8,s4_m01_baseline_arch,stage2,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s4_m01_baseline_arch,skipped_inconclusive,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Run s4_m01_baseline_arch requires template_fro...
9,s4_m02_dropout_030,stage2,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s4_m02_dropout_030,skipped_inconclusive,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Run s4_m02_dropout_030 requires template_from_...



[STATUS COUNTS]


,count
terminal_status,
skipped_inconclusive,14
failed_experiment,8
skipped_gate,2



[GATE DECISION]
{
  "generated_at": "2026-02-27T15:16:33.587517",
  "gate_pass": false,
  "reason": "stage3_no_candidate",
  "adaptive_recall_target": 0.4,
  "adaptive_target_reasoning": "max(0.4000, baseline(0.3442)+0.0500)",
  "best_stage3_run_id": null,
  "best_stage3_metrics": null,
  "pivot_recommendation": "USAD",
  "stage_validity": {
    "stage1": {
      "required": 5,
      "valid_count": 0,
      "passed": false,
      "status": "inconclusive"
    },
    "stage2": {
      "required": 6,
      "valid_count": 0,
      "passed": false,
      "status": "inconclusive"
    },
    "stage3": {
      "required": 5,
      "valid_count": 0,
      "passed": false,
      "status": "inconclusive"
    },
    "stage4": {
      "required": 2,
      "valid_count": 0,
      "passed": true,
      "status": "skipped_by_design"
    }
  },
  "sprint_history_summary": {
    "recall_p50": null,
    "recall_p90": null,
    "best_recall_under_guardrail": null,
    "best_recall_path": "",
    "count_a

# RESEARCH REPORT CSE F1 - SPRINT 4

- generated_at: 2026-02-27T15:16:33.593252
- source_summary: `/content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv`
- gate_decision: `/content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json`

## Ringkasan / Summary

- Objective: maximize CSE recall dengan guardrail CIC FPR rendah.
- Objective (EN): maximize CSE recall under low CIC-FPR guardrail.
- Gate pass: `False` (reason: `stage3_no_candidate`).
- Adaptive recall target: `0.4` (max(0.4000, baseline(0.3442)+0.0500)).

## Stage Validity

| stage | required_valid_runs | valid_runs | status |
| --- | ---: | ---: | --- |
| stage1 | 5 | 0 | inconclusive |
| stage2 | 6 | 0 | inconclusive |
| stage3 | 5 | 0 | inconclusive |
| stage4 | 2 | 0 | skipped_by_design |

## Best Candidate Per Stage

| stage | run_id | mode | cse_recall | cse_precision | cse_f1 | cic_fpr |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| stage0 | - | - | - | - | - | - |
| stage1 | - | - | - | - | - | - |
| stage2 | - | - | - | - | - | - |
| stage3 | - | - | - | - | - | - |
| stage4 | - | - | - | - | - | - |

## Operational Notes

- ID Primary: Terminologi teknis tetap English untuk konsistensi.
- EN Mirror: Technical keywords are intentionally kept in English.

## Run Status

- total_runs: 24
- success: 0
- failed_experiment: 8
- failed_infra_exhausted: 0
- pending_or_skipped: 16



[RUNTIME STATUS JSON]
{
  "s4_p01_robust_q995_clip20_corr090": {
    "retry_count": 0,
    "terminal_status": "failed_experiment",
    "last_error": "Command failed rc=1 after 1.52s: /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_p01_robust_q995_clip20_corr090.yaml",
    "hash_violation": false,
    "wall_time_sec": 0.0,
    "attempt_logs": [
      {
        "ts": "2026-02-27T15:16:24.383200",
        "kind": "experiment",
        "error": "Command failed rc=1 after 1.52s: /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_p01_robust_q995_clip20_corr090.yaml"
      }
    ]
  },
  "s4_p02_robust_q997_clip20_corr090": {
    "retry_count": 0,
    "terminal_status": "failed_experiment",
    "last_error": "Command failed rc=1 after 1.53s: /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/rese

## 4) Resume Guide

Kalau runtime Colab putus:
1. Run lagi dari cell mount + clone/pull + symlink.
2. Jalankan stage berikutnya atau stage yang sama.
3. `--skip-existing` akan melanjutkan dari artifact yang sudah ada.